<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-11-self-hosting/lesson-11.3-hybrid-litellm/practice/GCP_Capstone_11.3_Practice_Lab.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practice Lab 11.3 — Hybrid LiteLLM Gateway

Runnable companion to the published practice lab: every exercise with a complete solution grounded in the lesson notebook. Run the **Setup** cell first, then work through the exercises. Cloud Shell / `gcloud` steps are `%%bash` cells.

---

## Setup: install, authenticate, and set project constants

Run this first. It installs LiteLLM + Presidio + the Google Cloud SDKs, authenticates via Application Default Credentials (Colab), and defines the project/region constants every exercise below reuses. No API keys anywhere — the gateway and Vertex AI both use ADC.

In [ ]:
%%bash
pip install -q \
  'litellm[proxy]>=1.83' \
  'presidio-analyzer==2.2.*' 'presidio-anonymizer==2.2.*' \
  'google-cloud-dlp>=3.24.0' 'google-cloud-bigquery>=3.25.0' \
  'google-genai>=1.0.0' 'openai>=1.40'
python -m spacy download en_core_web_lg -q

In [ ]:
# Application Default Credentials (runs only on Colab; skipped locally)
try:
    from google.colab import auth
    auth.authenticate_user()
    print('Authenticated via Colab ADC')
except ImportError:
    print('Not on Colab — assuming ADC already configured (gcloud auth application-default login)')

# Shared constants used across every exercise
PROJECT_ID = 'documind-ai-YOUR-ID'   # <-- replace with your project id
REGION = 'us-central1'               # asia-south1 for India production
USD_INR = 85                          # for any INR cost display
print(f'Project: {PROJECT_ID} | Region: {REGION}')

## Exercise 1: config.yaml with 3 models

**Difficulty:** Easy

Write `model_list` with documind-sensitive (Gemma), documind-general (Flash), documind-reasoning (Pro). Include tags arrays and rpm/tpm limits.

1. Define three models: `documind-sensitive` (self-hosted Gemma via `hosted_vllm`), `documind-general` (`vertex_ai/gemini-3.6-flash`), `documind-reasoning` (`vertex_ai/gemini-3.1-pro-preview`).
2. Give each a `tags` array so LiteLLM can tag-route.
3. Add `rpm`/`tpm` limits and input/output cost-per-token on the self-hosted model.
4. Write the whole thing to `config.yaml`.

*Expected behaviour:* YAML with hosted_vllm + vertex_ai providers, tags for routing, input/output cost per token.

In [ ]:
# See Colab Cell 1 — the authoritative config. This single string carries every
# piece the later exercises reference (models, fallbacks, guardrail, budgets),
# so we write it once and each exercise inspects the relevant block.
CONFIG_YAML = '''
model_list:
  - model_name: documind-sensitive
    litellm_params:
      model: hosted_vllm/google/gemma-3-4b-it
      api_base: https://gemma-vllm-xxx-uc.a.run.app/v1
      api_key: "internal-token"
      rpm: 60
      tpm: 100000
      tags: ["sensitive", "hipaa", "gdpr", "dpdpa"]
      input_cost_per_token: 0.00001
      output_cost_per_token: 0.00001

  - model_name: documind-general
    litellm_params:
      model: vertex_ai/gemini-3.6-flash
      vertex_project: os.environ/VERTEXAI_PROJECT
      vertex_location: global
      tags: ["general", "public"]

  - model_name: documind-reasoning
    litellm_params:
      model: vertex_ai/gemini-3.1-pro-preview
      vertex_project: os.environ/VERTEXAI_PROJECT
      vertex_location: global
      tags: ["complex", "reasoning"]

router_settings:
  routing_strategy: simple-shuffle
  enable_tag_filtering: true
  fallbacks:
    - documind-sensitive: ["documind-general"]
    - documind-general: ["documind-sensitive"]
    - documind-reasoning: ["documind-general"]
  num_retries: 3
  timeout: 30
  allowed_fails: 3
  cooldown_time: 60
  retry_policy:
    RateLimitErrorRetries: 3
    TimeoutErrorRetries: 2
    InternalServerErrorRetries: 2
    AuthenticationErrorRetries: 0

guardrails:
  - guardrail_name: "documind-pii-router"
    litellm_params:
      guardrail: custom
      mode: "pre_call"
      callback_class: documind_router.DocuMindRouter

litellm_settings:
  callbacks: ["langfuse_otel"]
  langfuse_default_tags: ["environment", "model", "tenant"]

general_settings:
  master_key: os.environ/LITELLM_MASTER_KEY
  database_url: os.environ/DATABASE_URL

tag_budget_config:
  tenant-acme:
    max_budget: 500.00
    budget_duration: "30d"
  tenant-enterprise:
    max_budget: 5000.00
    budget_duration: "30d"
'''
with open('config.yaml', 'w') as f:
    f.write(CONFIG_YAML)
print('config.yaml written')
print('Tag-based routing + bidirectional fallback + per-tenant budgets')

## Exercise 2: Deploy LiteLLM to Cloud Run

**Difficulty:** Easy

Deploy `docker.litellm.ai/berriai/litellm:main-stable` as a CPU-only Cloud Run service. Config from a GCS bucket. Master key from Secret Manager.

1. Build a Dockerfile on the official LiteLLM image, adding Presidio + Google Cloud SDKs and the spaCy model.
2. Upload `config.yaml` to a GCS config bucket.
3. `gcloud run deploy` with `--cpu 1 --memory 2Gi --min-instances 1`, master key + DB URL from Secret Manager.
4. Grant the service account Vertex AI, run.invoker, and DLP roles.

*Expected behaviour:* gcloud run deploy with --cpu=1 --memory=2Gi --min-instances=1 + IAM roles for Vertex AI and internal invoker.

In [ ]:
# See Colab Cell 5 — Dockerfile that layers Presidio + GCP SDKs onto LiteLLM.
DOCKERFILE = '''
FROM docker.litellm.ai/berriai/litellm:main-stable

WORKDIR /app

# Install Presidio + Google Cloud SDKs for custom guardrail
RUN pip install --no-cache-dir \\
    presidio-analyzer==2.2.* \\
    presidio-anonymizer==2.2.* \\
    google-cloud-dlp>=3.24.0 \\
    google-cloud-bigquery>=3.25.0

# Download spaCy model for Presidio NER
RUN python -m spacy download en_core_web_lg

# Copy custom guardrail code
COPY documind_classifier.py documind_router.py dlp_audit.py ./

# config.yaml loads from GCS at runtime (LITELLM_CONFIG_BUCKET env)
ENV PORT=8080
EXPOSE 8080

CMD ["--config", "/app/config.yaml", "--port", "8080", "--host", "0.0.0.0"]
'''
with open('Dockerfile', 'w') as f:
    f.write(DOCKERFILE)
print('Dockerfile written')

In [ ]:
%%bash
# See Colab Cell 5 — deploy the gateway. PROJECT_ID is your project id.
export PROJECT_ID=documind-ai-YOUR-ID

# Create config bucket + upload config.yaml
gsutil mb gs://$PROJECT_ID-litellm-config
gsutil cp config.yaml gs://$PROJECT_ID-litellm-config/config.yaml

# Create Postgres for state (Cloud SQL)
gcloud sql instances create litellm-db \
  --database-version=POSTGRES_15 \
  --cpu=1 --memory=4GB \
  --region=us-central1 \
  --network=default

# Deploy LiteLLM proxy to Cloud Run (CPU-only, warm min instance)
# PREREQUISITE (one-time, real GCP): create the litellm service account, the master-key and
# db-url secrets, and the VPC connector referenced below before this deploy:
#   gcloud iam service-accounts create litellm-sa
#   gcloud secrets create litellm-master --data-file=- ; gcloud secrets create litellm-db-url --data-file=-
gcloud run deploy litellm-gateway \
  --source . \
  --region us-central1 \
  --cpu 1 --memory 2Gi \
  --port 8080 \
  --min-instances 1 \
  --max-instances 5 \
  --concurrency 100 \
  --service-account litellm-sa@$PROJECT_ID.iam.gserviceaccount.com \
  --set-env-vars="LITELLM_CONFIG_BUCKET_TYPE=gcs,LITELLM_CONFIG_BUCKET=$PROJECT_ID-litellm-config,VERTEXAI_PROJECT=$PROJECT_ID" \
  --set-secrets="LITELLM_MASTER_KEY=litellm-master:latest,DATABASE_URL=litellm-db-url:latest" \
  --vpc-egress=private-ranges-only \
  --network=documind-vpc \
  --subnet=documind-subnet

# Grant IAM permissions
gcloud projects add-iam-policy-binding $PROJECT_ID \
  --member="serviceAccount:litellm-sa@$PROJECT_ID.iam.gserviceaccount.com" \
  --role="roles/aiplatform.user"
gcloud projects add-iam-policy-binding $PROJECT_ID \
  --member="serviceAccount:litellm-sa@$PROJECT_ID.iam.gserviceaccount.com" \
  --role="roles/run.invoker"
gcloud projects add-iam-policy-binding $PROJECT_ID \
  --member="serviceAccount:litellm-sa@$PROJECT_ID.iam.gserviceaccount.com" \
  --role="roles/dlp.user"

## Exercise 3: Bidirectional fallback config

**Difficulty:** Easy

`router_settings.fallbacks` in both directions. `allowed_fails=3` + `cooldown_time=60` + `retry_policy` per error type.

1. Configure `documind-sensitive ↔ documind-general` as a two-way fallback pair.
2. Add `allowed_fails: 3` and `cooldown_time: 60` so a flaky deployment is briefly benched, not hammered.
3. Set a per-error-type `retry_policy` — and crucially `AuthenticationErrorRetries: 0` (never retry auth failures).

*Expected behaviour:* documind-sensitive ↔ documind-general bidirectional. AuthenticationErrorRetries: 0 (never retry auth).

In [ ]:
# See Step 9 YAML — the router_settings block already lives inside the config.yaml
# from Exercise 1. Here we parse it back out and assert the fallback is genuinely
# bidirectional and that auth errors are never retried.
import yaml

cfg = yaml.safe_load(CONFIG_YAML)
rs = cfg['router_settings']

fallbacks = {list(d.keys())[0]: list(d.values())[0] for d in rs['fallbacks']}
print('Fallback map:')
for src, dests in fallbacks.items():
    print(f'  {src} -> {dests}')

# Bidirectional check
assert 'documind-general' in fallbacks['documind-sensitive']
assert 'documind-sensitive' in fallbacks['documind-general']
print('\\nBidirectional sensitive <-> general: OK')

# Circuit-breaker + retry policy
assert rs['allowed_fails'] == 3 and rs['cooldown_time'] == 60
assert rs['retry_policy']['AuthenticationErrorRetries'] == 0
print(f"allowed_fails={rs['allowed_fails']} cooldown_time={rs['cooldown_time']}s")
print('retry_policy:', rs['retry_policy'])
print('AuthenticationErrorRetries == 0 (never retry auth): OK')

## Exercise 4: PII classifier regex + Presidio

**Difficulty:** Medium

`classify_tier()` returning PUBLIC/INTERNAL/CONFIDENTIAL/RESTRICTED. Test with Aadhaar, PAN, SSN, email, and plain-text samples. Measure latency.

1. Layer 1: fast regex for RESTRICTED identifiers (SSN, Aadhaar, PAN, credit card) and CONFIDENTIAL ones (email, phone).
2. Layer 2: confirm RESTRICTED hits with Presidio's `AnalyzerEngine` (threshold 0.7) to cut false positives.
3. Return the highest matching `SensitivityTier`.
4. Run the test cases and time the classifier — target <50ms on typical prompts.

*Expected behaviour:* Regex patterns + Presidio AnalyzerEngine with threshold 0.7. <50ms on typical prompts.

In [ ]:
# See Colab Cell 2 — write the classifier module (imported by the router in Ex5).
CLASSIFIER_PY = r'''
import re
from enum import IntEnum
from presidio_analyzer import AnalyzerEngine

class SensitivityTier(IntEnum):
    PUBLIC = 0
    INTERNAL = 1
    CONFIDENTIAL = 2
    RESTRICTED = 3

PATTERNS_RESTRICTED = {
    "SSN": re.compile(r"\b(?!000|666|9\d{2})\d{3}[-\s]?\d{2}[-\s]?\d{4}\b"),
    "AADHAAR": re.compile(r"\b[2-9]\d{3}[-\s]?\d{4}[-\s]?\d{4}\b"),
    "PAN": re.compile(r"\b[A-Z]{5}[0-9]{4}[A-Z]\b"),
    "CREDIT_CARD": re.compile(r"\b(?:\d[ -]*?){13,19}\b"),
}
PATTERNS_CONFIDENTIAL = {
    "EMAIL": re.compile(r"\b[\w.-]+@[\w.-]+\.\w+\b"),
    "PHONE": re.compile(r"\b\d{3}[-.]?\d{3}[-.]?\d{4}\b"),
}

_analyzer = None
def get_analyzer():
    global _analyzer
    if _analyzer is None:
        _analyzer = AnalyzerEngine()
    return _analyzer

def classify_tier(text: str) -> SensitivityTier:
    # Layer 1: fast regex for RESTRICTED
    for name, pattern in PATTERNS_RESTRICTED.items():
        if pattern.search(text):
            # Layer 2: Presidio confirms context-aware
            results = get_analyzer().analyze(text=text, language="en")
            if any(r.score >= 0.7 for r in results):
                return SensitivityTier.RESTRICTED
    # Layer 1 for CONFIDENTIAL
    for name, pattern in PATTERNS_CONFIDENTIAL.items():
        if pattern.search(text):
            return SensitivityTier.CONFIDENTIAL
    return SensitivityTier.PUBLIC
'''
with open('documind_classifier.py', 'w') as f:
    f.write(CLASSIFIER_PY)
print('documind_classifier.py written')

In [ ]:
# Import the module we just wrote, run the test cases, and measure latency.
import time
from documind_classifier import classify_tier, SensitivityTier

test_cases = [
    ("What is GDPR?", SensitivityTier.PUBLIC),
    ("Email me at user@acme.com", SensitivityTier.CONFIDENTIAL),
    ("My Aadhaar is 2345-6789-0123", SensitivityTier.RESTRICTED),
    ("SSN 123-45-6789", SensitivityTier.RESTRICTED),
    ("PAN ABCDE1234F", SensitivityTier.RESTRICTED),
]

get_analyzer_warm = classify_tier("warm up presidio")  # first call loads the model

for text, expected in test_cases:
    t0 = time.perf_counter()
    got = classify_tier(text)
    ms = (time.perf_counter() - t0) * 1000
    flag = 'OK' if got == expected else 'MISMATCH'
    print(f'[{flag}] {got.name:12s} ({ms:5.1f} ms)  <- {text!r}')

## Exercise 5: CustomGuardrail pre-call hook

**Difficulty:** Medium

`DocuMindRouter` subclass that sets `data["model"]` based on the classified tier. Register it in the config.yaml `guardrails` section.

1. Subclass LiteLLM's `CustomGuardrail`; build Presidio analyzer + anonymizer in `__init__`.
2. In `async_pre_call_hook`, concatenate user messages and classify the tier.
3. RESTRICTED → force `documind-sensitive` (self-hosted); CONFIDENTIAL → mask PII then send to `documind-general`; PUBLIC → leave as-is.
4. Stamp `routing_tier` and `original_model` into `metadata` for the audit trail.
5. Register under `guardrails:` with `mode: pre_call` (already present in config.yaml).

*Expected behaviour:* async async_pre_call_hook that classifies, masks if CONFIDENTIAL, sets model if RESTRICTED, logs routing_tier to metadata.

In [ ]:
# See Colab Cell 3 — the pre-call guardrail that actually re-routes.
ROUTER_PY = '''
from litellm.integrations.custom_guardrail import CustomGuardrail
from documind_classifier import classify_tier, SensitivityTier
from presidio_analyzer import AnalyzerEngine
from presidio_anonymizer import AnonymizerEngine
import logging

logger = logging.getLogger("documind.router")

class DocuMindRouter(CustomGuardrail):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.analyzer = AnalyzerEngine()
        self.anonymizer = AnonymizerEngine()

    def mask_pii(self, data: dict) -> dict:
        """Replace PII with placeholders before sending to external API."""
        for msg in data.get("messages", []):
            content = msg.get("content")
            if not isinstance(content, str):
                continue
            results = self.analyzer.analyze(text=content, language="en")
            if results:
                anonymized = self.anonymizer.anonymize(text=content, analyzer_results=results)
                msg["content"] = anonymized.text
        return data

    async def async_pre_call_hook(self, user_api_key_dict, cache, data, call_type):
        """Runs BEFORE LiteLLM dispatches the request."""
        # Concatenate all user messages
        text = " ".join(
            m.get("content", "") for m in data.get("messages", [])
            if isinstance(m.get("content"), str)
        )

        tier = classify_tier(text)
        original_model = data.get("model")

        # Routing decision
        if tier == SensitivityTier.RESTRICTED:
            data["model"] = "documind-sensitive"
            logger.info(f"RESTRICTED routed to self-hosted (was {original_model})")
        elif tier == SensitivityTier.CONFIDENTIAL:
            data = self.mask_pii(data)
            data["model"] = "documind-general"
            logger.info(f"CONFIDENTIAL masked and routed to general")
        # else PUBLIC: keep original model selection

        # Log routing decision for audit trail
        data["metadata"] = data.get("metadata", {})
        data["metadata"]["routing_tier"] = tier.name
        data["metadata"]["original_model"] = original_model
        return data
'''
with open('documind_router.py', 'w') as f:
    f.write(ROUTER_PY)
print('documind_router.py written')
print('Registered in config.yaml under guardrails: with mode: pre_call')

## Exercise 6: Cloud DLP async audit

**Difficulty:** Medium

Create an inspect template with India infoTypes. A `BackgroundTask` sends a 10% sample to DLP and writes findings to a BigQuery audit table.

1. Build an inspect template including `INDIA_AADHAAR_INDIVIDUAL`, `INDIA_PAN_INDIVIDUAL`, `INDIA_GST_INDIVIDUAL`, plus SSN / credit card / medical.
2. Fire-and-forget: only 10% of requests get DLP-scanned (cost control).
3. Derive a `dlp_tier` from the findings and compare it to the classifier's `classified_tier`.
4. Insert one row per audited request into `<project>.documind.routing_audit`.

*Expected behaviour:* Template with INDIA_AADHAAR_INDIVIDUAL, INDIA_PAN_INDIVIDUAL. 10% sampling. BigQuery schema with classified_tier vs dlp_tier match.

In [ ]:
# See Colab Cell 4 — async DLP audit layer that grades the classifier.
DLP_AUDIT_PY = '''
from google.cloud import dlp_v2, bigquery
import json, datetime, random, asyncio
import logging
logger = logging.getLogger("documind.dlp_audit")

dlp_client = dlp_v2.DlpServiceClient()
bq_client = bigquery.Client()
PROJECT_ID = "your-project-id"
DLP_TEMPLATE = f"projects/{PROJECT_ID}/locations/global/inspectTemplates/documind-pii"

def create_documind_inspect_template():
    """One-time setup: create inspect template with India info types."""
    template = {
        "inspect_config": {
            "info_types": [
                {"name": "EMAIL_ADDRESS"},
                {"name": "PHONE_NUMBER"},
                {"name": "CREDIT_CARD_NUMBER"},
                {"name": "US_SOCIAL_SECURITY_NUMBER"},
                {"name": "INDIA_AADHAAR_INDIVIDUAL"},
                {"name": "INDIA_PAN_INDIVIDUAL"},
                {"name": "INDIA_GST_INDIVIDUAL"},
                {"name": "PERSON_NAME"},
                {"name": "MEDICAL_TERM"},
                {"name": "DATE_OF_BIRTH"},
            ],
            "min_likelihood": dlp_v2.Likelihood.POSSIBLE,
            "include_quote": False,  # Don't include raw PII in findings
        },
        "display_name": "DocuMind PII Audit Template",
        "description": "Audit layer for hybrid routing validation",
    }
    return dlp_client.create_inspect_template(
        request={
            "parent": f"projects/{PROJECT_ID}/locations/global",
            "inspect_template": template,
            "template_id": "documind-pii"
        }
    )

async def async_audit_to_bigquery(request_id: str, text: str,
                                    classified_tier: str,
                                    sample_rate: float = 0.1):
    """Fire-and-forget DLP scan + BigQuery log.
    Only 10% of requests get DLP scanned (cost control).
    Findings compared against classified_tier to measure accuracy."""
    if random.random() > sample_rate:
        return

    try:
        # Scan with Cloud DLP
        response = await asyncio.to_thread(
            dlp_client.inspect_content,
            request={
                "parent": f"projects/{PROJECT_ID}/locations/global",
                "inspect_template_name": DLP_TEMPLATE,
                "item": {"value": text[:10000]}  # DLP has size limits
            }
        )

        findings = response.result.findings
        dlp_entities = [f.info_type.name for f in findings]

        # Determine DLP-inferred tier
        restricted_types = {"INDIA_AADHAAR_INDIVIDUAL", "INDIA_PAN_INDIVIDUAL",
                            "CREDIT_CARD_NUMBER", "US_SOCIAL_SECURITY_NUMBER",
                            "MEDICAL_TERM"}
        dlp_tier = ("RESTRICTED" if any(e in restricted_types for e in dlp_entities)
                    else "CONFIDENTIAL" if dlp_entities
                    else "PUBLIC")

        # Log for accuracy measurement
        bq_client.insert_rows_json(
            f"{PROJECT_ID}.documind.routing_audit",
            [{
                "request_id": request_id,
                "classified_tier": classified_tier,
                "dlp_tier": dlp_tier,
                "dlp_entities": dlp_entities,
                "match": classified_tier == dlp_tier,
                "timestamp": datetime.datetime.utcnow().isoformat(),
            }]
        )
    except Exception as e:
        logger.error(f"DLP audit failed: {e}")
'''
with open('dlp_audit.py', 'w') as f:
    f.write(DLP_AUDIT_PY)
print('dlp_audit.py written')
print('Async audit layer: 10% sample rate, ~$3/GB scanned')
print('BigQuery rollup reveals classifier accuracy against DLP ground truth')

## Exercise 7: Per-tenant budget enforcement

**Difficulty:** Challenge

Virtual keys per tenant via the `/key/generate` endpoint + `tag_budget_config`. Test that budget exhaustion returns 429 with a `Retry-After` header.

1. Confirm `tag_budget_config` in config.yaml caps `tenant-acme` at $500/30d and `tenant-enterprise` at $5000/30d (Step 10 YAML).
2. Mint a virtual key per tenant against `/key/generate`, scoped to the tenant tag.
3. Send requests carrying that key's tenant tag; when the tag budget is exhausted the gateway returns HTTP 429.
4. Verify the 429 carries a `Retry-After` header and that the budget resets per `budget_duration`.

*Expected behaviour:* Multiple virtual keys scoped to tenant tags. Exceeding budget returns HTTP 429. Budget resets per duration (30d/1mo).

In [ ]:
# See Step 10 YAML — inspect the per-tenant budget block from config.yaml.
import yaml
budgets = yaml.safe_load(CONFIG_YAML)['tag_budget_config']
for tenant, b in budgets.items():
    inr = b['max_budget'] * USD_INR
    print(f"{tenant:18s} max ${b['max_budget']:>8.2f} (Rs {inr:>10,.0f}) / {b['budget_duration']}")

In [ ]:
%%bash
# See Cell 6 (client side) — mint a virtual key per tenant, scoped to the tenant
# tag, against the running gateway's /key/generate admin endpoint. The master key
# comes from Secret Manager (never hardcode it).
export GATEWAY_URL=https://litellm-gateway-xxxxx-uc.a.run.app
export MASTER_KEY=$(gcloud secrets versions access latest --secret=litellm-master)

# tenant-acme virtual key, tagged so tag_budget_config applies
curl -sS -X POST "$GATEWAY_URL/key/generate" \
  -H "Authorization: Bearer $MASTER_KEY" \
  -H "Content-Type: application/json" \
  -d '{
        "key_alias": "acme-prod",
        "tags": ["tenant-acme"],
        "models": ["documind-general", "documind-sensitive", "documind-reasoning"],
        "max_budget": 500.00,
        "budget_duration": "30d"
      }'

# tenant-enterprise virtual key
curl -sS -X POST "$GATEWAY_URL/key/generate" \
  -H "Authorization: Bearer $MASTER_KEY" \
  -H "Content-Type: application/json" \
  -d '{
        "key_alias": "enterprise-prod",
        "tags": ["tenant-enterprise"],
        "models": ["documind-general", "documind-sensitive", "documind-reasoning"],
        "max_budget": 5000.00,
        "budget_duration": "30d"
      }'

In [ ]:
# Verify budget-exhaustion behaviour: once a tenant's tag budget is spent, the
# gateway rejects with HTTP 429 + Retry-After. This drives the live gateway via
# the standard OpenAI SDK (base_url -> LiteLLM). Fill in a real key to run.
import openai

GATEWAY_URL = 'https://litellm-gateway-xxxxx-uc.a.run.app'
VIRTUAL_KEY = 'sk-litellm-virtual-key-for-tenant-acme'  # from /key/generate above

client = openai.OpenAI(base_url=f'{GATEWAY_URL}/v1', api_key=VIRTUAL_KEY)

try:
    client.chat.completions.create(
        model='documind-general',
        messages=[{'role': 'user', 'content': 'ping'}],
        extra_body={'metadata': {'tags': ['tenant-acme']}},
    )
    print('Request accepted — tenant is under budget')
except openai.RateLimitError as e:
    # LiteLLM returns 429 when the tag budget is exhausted
    retry_after = e.response.headers.get('retry-after')
    print(f'HTTP 429 budget exceeded. Retry-After: {retry_after}s')
    print('Budget resets per budget_duration (30d).')

## Exercise 8: Cost dashboard with savings

**Difficulty:** Challenge

BigQuery query joining LiteLLM `spend_logs` + Cloud Run billing. Compute the counterfactual all-Pro cost vs the actual hybrid cost. Expected: 6–8x savings.

1. Break down daily cost by model and per-tenant spend from `spend_logs`.
2. Show routing-tier distribution (compliance audit).
3. QUERY 4 is the ROI: `SUM(total_tokens * $0.000011)` (all-Pro counterfactual) vs `SUM(response_cost)` (actual), and a `savings_pct`.
4. QUERY 5 grades the classifier against DLP ground truth.

*Expected behaviour:* QUERY 4 reveals the ROI. SUM(total_tokens * $0.000011) vs SUM(response_cost). Show savings_pct > 85%.

In [ ]:
# See Colab Cell 7 — the dashboard SQL. Substitute your project id and run each
# query in BigQuery (or via bq_client.query below).
DASHBOARD_SQL = '''
-- LiteLLM spend logs land in the Postgres spend DB (store_model_in_db: true);
-- schedule a BigQuery export / scheduled query to populate the tables below
-- when general_settings.store_model_in_db: true is set in config.yaml

-- QUERY 1: Daily cost breakdown by model
SELECT
  DATE(start_time) AS date,
  model,
  COUNT(*) AS requests,
  SUM(total_tokens) AS tokens,
  ROUND(SUM(response_cost), 2) AS cost_usd
FROM `${PROJECT_ID}.litellm.spend_logs`
WHERE DATE(start_time) >= CURRENT_DATE() - 30
GROUP BY date, model
ORDER BY date DESC, cost_usd DESC;

-- QUERY 2: Per-tenant spend (metadata tags)
SELECT
  JSON_VALUE(metadata, '$.tags[0]') AS tenant,
  model,
  COUNT(*) AS requests,
  ROUND(SUM(response_cost), 2) AS cost_usd
FROM `${PROJECT_ID}.litellm.spend_logs`
WHERE DATE(start_time) >= CURRENT_DATE() - 30
GROUP BY tenant, model
ORDER BY cost_usd DESC;

-- QUERY 3: Routing tier distribution (compliance audit)
SELECT
  JSON_VALUE(metadata, '$.routing_tier') AS tier,
  model,
  COUNT(*) AS requests,
  ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 2) AS pct_of_traffic
FROM `${PROJECT_ID}.litellm.spend_logs`
WHERE DATE(start_time) >= CURRENT_DATE() - 7
GROUP BY tier, model
ORDER BY tier, requests DESC;

-- QUERY 4: ROUTING SAVINGS -- counterfactual all-Pro cost
SELECT
  SUM(response_cost) AS actual_hybrid_cost,
  SUM(total_tokens * 0.000011) AS counterfactual_all_pro_cost,
  SUM(total_tokens * 0.000011) - SUM(response_cost) AS savings_usd,
  ROUND(100.0 * (1 - SUM(response_cost) / SUM(total_tokens * 0.000011)), 1) AS savings_pct
FROM `${PROJECT_ID}.litellm.spend_logs`
WHERE DATE(start_time) >= CURRENT_DATE() - 30;

-- QUERY 5: Classifier accuracy vs DLP ground truth
SELECT
  classified_tier,
  dlp_tier,
  COUNT(*) AS n,
  ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY classified_tier), 1) AS pct
FROM `${PROJECT_ID}.documind.routing_audit`
WHERE DATE(timestamp) >= CURRENT_DATE() - 7
GROUP BY classified_tier, dlp_tier
ORDER BY classified_tier, n DESC;
'''
print(DASHBOARD_SQL.replace('${PROJECT_ID}', PROJECT_ID))
print()
print('Typical DocuMind results (30d):')
actual, all_pro = 1113, 7932
savings_pct = round(100 * (1 - actual / all_pro), 1)
print(f'  actual_hybrid_cost:      ${actual:,}  (Rs {actual*USD_INR:,})')
print(f'  counterfactual_all_pro:  ${all_pro:,}  (Rs {all_pro*USD_INR:,})')
print(f'  savings_pct:             {savings_pct}%  ({all_pro/actual:.1f}x cheaper)')
print('Classifier accuracy target: >95% match with DLP for RESTRICTED tier')